# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates loading and exploring the FAIR\(^2\) dataset in Croissant format using the `mlcroissant` library.

### Dataset Source
The dataset is available via the Croissant schema URL below.

In [ ]:
# Ensure mlcroissant is installed (uncomment and run if needed)
!pip install mlcroissant

## 1. Data Loading
We load the FAIR\(^2\) mass spectrometry dataset, including both the metadata and the records, using the `mlcroissant` package.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant metadata URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant Dataset
dataset = mlc.Dataset(croissant_url)

# Print title and description from metadata
print(f"Dataset: {dataset.metadata.name}\n")
print(f"Description: {dataset.metadata.description}\n")

## 2. Data Overview
Let's list available Record Sets, their `@id`s, and their Fields (`@id`s and names). This is essential to know what entities we can extract and analyze.

**Note: All Croissant entities are referenced by their `@id`.**

In [ ]:
# List all record sets and their fields (by @id)
print("Available Record Sets:\n======================")
for recset in dataset.metadata.record_sets:
    print(f"Record Set: {recset.name}")
    print(f"  @id: {recset.id}")
    print("  Fields:")
    for field in recset.fields:
        print(f"    - {field.name} (@id: {field.id})")
    print()

## 3. Data Extraction
Here, we extract the data from a chosen record set using its `@id` as seen above. We'll list all record sets found, then load one of them (the main data table as an example).

All selection is done by `@id`.

In [ ]:
# Pick out all record sets' @id
record_set_ids = [recset.id for recset in dataset.metadata.record_sets]
print("Record set IDs to load:", record_set_ids)

dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records from RecordSet {record_set_id} ...")
    data = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(data)
    dataframes[record_set_id] = df

main_record_set_id = record_set_ids[0]  # Example: use the first record set
print(f"\nMain RecordSet columns: {dataframes[main_record_set_id].columns.tolist()}")
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)

We'll select a numeric field (by its `@id`) and demonstrate common data processing steps: filtering, normalization, and grouping.

All operations below strictly reference fields by their Croissant `@id`.

In [ ]:
# For demonstration, pick a numeric field from the main record set
# List all fields and pick one with numerical data, e.g. 'molecular_weight' if present

main_recset = None
for recset in dataset.metadata.record_sets:
    if recset.id == main_record_set_id:
        main_recset = recset
        break

print(f"Fields in {main_record_set_id}:")
for field in main_recset.fields:
    print(f"- {field.name}: @id={field.id}, datatype={getattr(field, 'data_type', None)}")

# We'll use the data field with @id equal to the molecular weight column if available, otherwise the first numeric field
numeric_field_id = None

for field in main_recset.fields:
    if hasattr(field, 'data_type') and field.data_type in ['schema:Float', 'schema:Integer', 'schema:Number']:
        if 'weight' in field.name.lower() or 'mw' in field.name.lower():
            numeric_field_id = field.id
            break
# Fallback: pick the first numeric field if none matched 'weight'/'mw'
if numeric_field_id is None:
    for field in main_recset.fields:
        if hasattr(field, 'data_type') and field.data_type in ['schema:Float', 'schema:Integer', 'schema:Number']:
            numeric_field_id = field.id
            break
if numeric_field_id is None:
    raise ValueError("No numeric field found in main recordset.")

print(f"\nSelected numeric field for EDA: {numeric_field_id}")

df = dataframes[main_record_set_id]

# Ensure numeric
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

threshold = df[numeric_field_id].mean()
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records where {numeric_field_id} > {threshold:.2f} (mean): {len(filtered_df)} records\n")
print(filtered_df.head())

# Normalize
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()

print(f"\nFirst few normalized values:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by a categorical field (try to find one, e.g., 'accession', 'description', etc.)
group_field_id = None
for field in main_recset.fields:
    if hasattr(field, 'data_type') and field.data_type in ['schema:Text', None, 'Text']:
        if 'accession' in field.name.lower() or 'description' in field.name.lower() or 'group' in field.name.lower():
            group_field_id = field.id
            break
# If none found, use the second column
if group_field_id is None and df.shape[1] > 1:
    group_field_id = df.columns[1]

print(f"\nGrouping by field: {group_field_id}\n")
if group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame('mean_value')
    print("Grouped means:")
    print(grouped_df.head())

## 5. Visualization

Let's visualize the distribution of the chosen numeric field and compare groups if possible.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(8,5))
sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# If grouping field exists, plot normalized values for each group (first 5 groups)
if group_field_id in filtered_df.columns:
    plt.figure(figsize=(10,6))
    top_groups = filtered_df[group_field_id].value_counts().nlargest(5).index
    sns.boxplot(x=group_field_id, y=f"{numeric_field_id}_normalized", data=filtered_df[filtered_df[group_field_id].isin(top_groups)])
    plt.title(f"{numeric_field_id}_normalized by {group_field_id} (Top 5 groups)")
    plt.xticks(rotation=30)
    plt.show()

## 6. Conclusion

In this notebook, we:
- Loaded a Croissant-described mass spectrometry dataset via `mlcroissant`.
- Listed all available record sets and fields (referenced by their `@id`).
- Extracted records using the record set `@id`.
- Performed EDA: filtering, normalization, grouping by categorical fields.
- Visualized the numeric field distribution and group comparisons.

Further analyses can focus on multi-record-set relationships, detailed field analysis, or integrating with other biological datasets. All Croissant elements were referenced and manipulated using their persistent `@id`s.